# Aula 08 · JSON, datas e exceções

Esta aula apresenta o [capítulo 8 do site](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/). A ideia central: **o dado de verdade chega
sujo**, e o programa precisa decidir o tamanho do estrago. JSON vira as estruturas
que você já conhece; data em texto vira data de verdade, que dá para subtrair; e o
`try/except` no lugar certo faz o coletor perder **uma linha**, não o arquivo.

**Ao fim da aula você consegue:**

1. ler um JSON com `json.loads` e navegar no dicionário e nas listas que saem dele;
2. converter texto em data com `strptime` e calcular durações;
3. tratar a linha ruim com `try/except` **por item**, contando o que foi descartado.

**Roteiro:** 📟 chamado · 🔥 aquecimento · 1. JSON · 2. datas · 3. exceções: o
coletor que não morre · 📟 resolvendo o chamado · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

## 📟 O chamado de hoje

> **Chamado #0826 — NOC Maré Net**
>
> *"Estagiário, o script que resume a coleta da madrugada **morreu de novo às
> 3h**: uma linha veio com o relógio corrompido e o programa inteiro parou. Ninguém
> soube de nada até a manhã. Preciso que ele **pule a linha ruim, conte quantas
> pulou** e diga quanto tempo durou a janela de alarmes da noite."*

No fim da aula você escreve o resumo que não morre.

## 🔥 Aquecimento — da aula passada

Sem rodar nada: o arquivo `alarmes.log` tem 3 alarmes e termina com uma linha em
branco. O que acontece?

```python
criticos = 0
with open("alarmes.log", encoding="utf-8") as arquivo:
    for linha in arquivo:
        if linha.split()[2] == "CRITICAL":
            criticos = criticos + 1
print(criticos)
```

<details>
<summary><b>Resposta</b></summary>

Dá `IndexError` na última volta — a linha em branco não tem `[2]` — e o `print`
nunca roda. Faltaram as duas linhas de defesa: `linha = linha.strip()` e
`if len(linha) == 0: continue`. Hoje você vê a outra forma de lidar com isso.

</details>

## 1. JSON

JSON é o formato de texto em que APIs e sistemas de gerência trocam dados — o
NetBox, a API de um roteador, o inventário de um provedor. `json.loads(texto)`
transforma o texto nas estruturas do Python:

| JSON | Python |
|---|---|
| `{ }` objeto | dicionário |
| `[ ]` array | lista |
| `"texto"` | `str` |
| `16`, `-21.4` | `int`, `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

📖 [capítulo 8 · JSON](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#json)

Rode a célula abaixo: é a resposta de um sistema de inventário, como texto.

In [ ]:
texto = '''
{
  "coleta": "2026-03-02T23:59:00",
  "equipamentos": [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "portas": 16, "em_servico": true},
    {"nome": "ONU-SUL-4512", "tipo": "ONU", "portas": 1, "em_servico": false}
  ]
}
'''

**✍️ Passo 1.** Escreva `import json`, faça `dados = json.loads(texto)` e imprima `type(dados)` e
`dados["coleta"]`.

In [ ]:
# ✍️ passo 1

**Preveja:** que tipo o `loads` devolve?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`<class 'dict'>` e `2026-03-02T23:59:00`. Depois do `loads` não há nada de novo: é
um **dicionário** da Aula 05. (A data continua sendo texto — o JSON não tem tipo de
data.)

</details>

**✍️ Passo 2.** Imprima `len(dados["equipamentos"])`, `dados["equipamentos"][0]["nome"]` e
`dados["equipamentos"][1]["em_servico"]`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que é `dados["equipamentos"]`? E o `false` do texto vira o quê?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`2`, `OLT-CENTRO-01` e `False`. `dados["equipamentos"]` é a **lista de dicionários**
da Aula 06, e o caminho até o nome se lê da esquerda para a direita: chave, posição,
chave. O `false` do JSON virou o `False` do Python.

</details>

**✍️ Passo 3.** O caminho de volta: crie `resumo = {"total": 2, "em_servico": 1}` e imprima
`json.dumps(resumo)` e `json.dumps(resumo, indent=2)`.

In [ ]:
# ✍️ passo 3

**Preveja:** o que o `dumps` devolve — um dicionário ou um texto?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Um **texto** JSON: o primeiro numa linha só, o segundo indentado, fácil de ler. É
assim que um script entrega resultado para outro sistema.

</details>

**✍️ Passo 4.** Tente ler um "JSON" com aspas simples: `json.loads("{'nome': 'OLT-CENTRO-01'}")`.

In [ ]:
# ✍️ passo 4

**Preveja:** o que há de errado neste JSON?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`JSONDecodeError`, reclamando de aspas duplas no nome da propriedade. JSON **só
aceita aspas duplas**, e booleano é `true`/`false` em minúsculas. A última linha da
mensagem diz exatamente isso — leia-a antes de qualquer coisa.

</details>

> ⚠️ **Armadilha.** Achar que JSON é "dicionário do Python escrito em texto". Parece, mas não é:
> `{'nome': 'OLT'}` e `{"ativo": True}` são Python válido e JSON **inválido**. Quem
> monta JSON à mão erra aqui; quem usa `json.dumps` não.

### 🎯 Sua vez — Quem está em serviço

Escreva `em_servico(texto_json)`, que recebe o texto JSON no formato acima e devolve
a **lista dos nomes** dos equipamentos com `em_servico` verdadeiro.

In [ ]:
import json


def em_servico(texto_json):
    # sua solução aqui
    pass

In [ ]:
confere(em_servico, [
    ((texto,), ["OLT-CENTRO-01"]),
    (('{"coleta": "x", "equipamentos": []}',), []),
    (('{"equipamentos": [{"nome": "A", "em_servico": true}, {"nome": "B", "em_servico": true}]}',),
     ["A", "B"]),
])

<details>
<summary><b>💡 Dica</b></summary>

`json.loads` primeiro; depois é o filtro da Aula 06 sobre `dados["equipamentos"]`.

</details>

<details>
<summary><b>✅ Uma solução</b></summary>

```python
import json


def em_servico(texto_json):
    dados = json.loads(texto_json)
    nomes = []
    for e in dados["equipamentos"]:
        if e["em_servico"]:
            nomes.append(e["nome"])
    return nomes
```

</details>

## 2. Datas

Data em texto é só texto: não dá para subtrair `"15:48:17" - "14:03:17"`. O módulo
`datetime` transforma o texto em **data de verdade** com `strptime` (*string parse
time*), seguindo um formato que você descreve: `%Y` ano, `%m` mês, `%d` dia, `%H`
hora, `%M` minuto, `%S` segundo.

📖 [capítulo 8 · Datas](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#datas)

**✍️ Passo 5.** Escreva `from datetime import datetime, timedelta`. Faça
`momento = datetime.strptime("2026-03-02 14:03:17", "%Y-%m-%d %H:%M:%S")` e imprima
`momento` e `momento.year, momento.hour`.

In [ ]:
# ✍️ passo 5

**Preveja:** o que o `strptime` faz com cada pedaço do formato?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`2026-03-02 14:03:17` e `2026 14`. O formato é um **molde** do texto: cada `%` diz o
que está naquela posição, e o resto (`-`, espaço, `:`) tem que bater literalmente.
Agora o ano e a hora são números que dá para usar.

</details>

**✍️ Passo 6.** Crie `inicio` e `fim` com `strptime`, para `"2026-03-02 14:03:17"` e
`"2026-03-02 15:48:17"`. Faça `duracao = fim - inicio` e imprima `duracao` e
`duracao.total_seconds() / 60`.

In [ ]:
# ✍️ passo 6

**Preveja:** a subtração de duas datas devolve um número ou outra coisa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Devolve uma **duração** (`timedelta`), que imprime `1:45:00`; `.total_seconds()` a
transforma em número, e dividindo por 60 sai `105.0` minutos. É essa subtração que
dá o tempo de reparo (MTTR) e a janela de indisponibilidade.

</details>

**✍️ Passo 7.** Imprima `inicio + timedelta(hours=2)` e, depois, `inicio.strftime("%d/%m/%Y")` e
`inicio.strftime("%H:%M")`.

In [ ]:
# ✍️ passo 7

**Preveja:** o que o `strftime` faz — o mesmo que o `strptime` ou o contrário?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Somar um `timedelta` desloca a data: `2026-03-02 16:03:17`. O `strftime` (*string
format time*) faz o **caminho inverso** do `strptime`: data para texto, no formato
que você quiser — `02/03/2026` e `14:03` para o relatório.

</details>

**✍️ Passo 8.** Compare como texto e como data: imprima `"31/12/2025" < "01/01/2026"`. Depois
converta as duas com `strptime(..., "%d/%m/%Y")` e imprima a mesma comparação entre
os `datetime`.

In [ ]:
# ✍️ passo 8

**Preveja:** 31 de dezembro vem antes de 1º de janeiro. As duas linhas concordam?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: o texto dá `False` e a data dá `True`. Texto se compara **caractere a
caractere**: `"3"` vem depois de `"0"`, e acabou. Só com `datetime` a comparação
entende o calendário.

</details>

> ⚠️ **Armadilha.** Confiar na comparação de texto porque "funcionou no log". Funciona no formato
> `AAAA-MM-DD`, em que a ordem alfabética coincide com a do calendário — e **só nele**.
> Com a data brasileira, dá errado em silêncio. E subtrair texto nunca funciona.

**✍️ Passo 9.** Tente ler uma data com o formato errado:
`datetime.strptime("02/03/2026", "%Y-%m-%d")`.

In [ ]:
# ✍️ passo 9

**Preveja:** qual é a mensagem quando o formato não descreve o texto?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`ValueError: time data '02/03/2026' does not match format '%Y-%m-%d'`. A mensagem é
literal: diz o texto e o formato que não casaram. Guarde o **tipo** do erro —
`ValueError` —, que ele volta no próximo bloco.

</details>

### 🎯 Sua vez — Minutos entre dois alarmes

Escreva `minutos_entre(inicio, fim)`, que recebe dois textos no formato
`"AAAA-MM-DD HH:MM:SS"` e devolve os minutos entre eles, **arredondados com 1
casa**.

In [ ]:
from datetime import datetime


def minutos_entre(inicio, fim):
    # sua solução aqui
    pass

In [ ]:
confere(minutos_entre, [
    (("2026-03-02 14:03:17", "2026-03-02 15:48:17"), 105.0),
    (("2026-03-02 23:50:00", "2026-03-03 00:10:30"), 20.5),
    (("2026-03-02 10:00:00", "2026-03-02 10:00:00"), 0.0),
])

<details>
<summary><b>💡 Dica</b></summary>

O segundo caso atravessa a meia-noite — por isso não dá para subtrair só as horas.
Converta os dois com `strptime`, subtraia, e use `.total_seconds() / 60`.

</details>

<details>
<summary><b>✅ Uma solução</b></summary>

```python
from datetime import datetime


def minutos_entre(inicio, fim):
    formato = "%Y-%m-%d %H:%M:%S"
    duracao = datetime.strptime(fim, formato) - datetime.strptime(inicio, formato)
    return round(duracao.total_seconds() / 60, 1)
```

</details>

## 3. Exceções: o coletor que não morre

Quando uma linha dá erro, o programa inteiro para. `try/except` diz ao Python: *tente
isto; se der este erro, faça aquilo em vez de parar*. A decisão importante não é
**se** usar, mas **onde** pôr o `try` — ela define o tamanho do estrago.

📖 [capítulo 8 · Exceções: o coletor que não morre](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#excecoes-o-coletor-que-nao-morre)

Rode a célula abaixo: quatro linhas, duas delas ruins.

In [ ]:
linhas = [
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal",
    "",
    "linha truncada",
    "2026-03-02 10:02:55 CRITICAL ONU-SUL-4512 sem resposta",
]

**✍️ Passo 10.** Conte os críticos com `try/except` **por item**: `criticos = 0` e `descartadas = 0`;
dentro do `for`, um `try:` com o `if linha.split()[2] == "CRITICAL":` que soma em
`criticos`, e um `except IndexError:` que soma em `descartadas`. Imprima os dois.

In [ ]:
# ✍️ passo 10

**Preveja:** quantos críticos e quantas linhas descartadas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`2` críticos e `2` descartadas. A linha vazia e a truncada deram `IndexError`; o
`except` registrou cada uma e o laço **continuou** para a próxima. Perdeu-se a linha
ruim, não o arquivo.

</details>

**✍️ Passo 11.** Agora mova o `try` para **fora** do laço: `criticos = 0`, depois `try:` com o `for`
inteiro dentro, e `except IndexError: pass`. Imprima `criticos`.

In [ ]:
# ✍️ passo 11

**Preveja:** quantos críticos esta versão conta?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Conta **1**. No primeiro erro (a linha vazia, a segunda), o Python saiu do `try` —
e, com ele, do laço inteiro. A quarta linha, que era crítica, nunca foi lida. E
nenhuma mensagem avisou.

</details>

> ⚠️ **Armadilha.** Achar que a posição do `try` é detalhe de estilo. É a decisão de projeto mais
> importante da aula: **em volta do item**, perde-se uma linha; **em volta do laço**,
> perde-se o resto do arquivo. E `except: pass` sem registrar nada transforma
> "processei 998 de 1000" em "processei tudo" — por isso contamos as descartadas.

**✍️ Passo 12.** O mesmo padrão numa função: escreva `para_numero(texto)` que, dentro de um `try:`,
faz `return round(float(texto), 2)` e, no `except ValueError:`, `return None`.
Imprima `para_numero("-21.456")`, `para_numero("sem leitura")` e `para_numero("")`.

In [ ]:
# ✍️ passo 12

**Preveja:** o que sai nas duas chamadas com texto que não é número?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`-21.46`, `None` e `None`. O `float` de um texto que não é número dá `ValueError`, e
a função o transforma num `None` que quem chamou pode testar. Repare no tipo do erro:
cada `except` diz **qual** erro espera — os outros continuam aparecendo, como devem.

</details>

### 🎯 Sua vez — Somar as leituras válidas

O medidor devolveu uma lista de textos, alguns inválidos. Escreva
`soma_leituras(textos)`, que devolve **dois valores**: a soma das leituras que são
números (arredondada com 2 casas) e quantas foram descartadas.

In [ ]:
def soma_leituras(textos):
    # sua solução aqui
    pass

In [ ]:
confere(soma_leituras, [
    ((["-21.4", "sem leitura", "-19.6"],), (-41.0, 1)),
    ((["-10", "", "erro", "x"],), (-10.0, 3)),
    (([],), (0, 0)),
])

<details>
<summary><b>💡 Dica</b></summary>

Um `try` **dentro** do laço, em volta do `float(texto)`; no `except ValueError`,
some 1 nas descartadas. No fim, `return round(soma, 2), descartadas`.

</details>

<details>
<summary><b>✅ Uma solução</b></summary>

```python
def soma_leituras(textos):
    soma = 0
    descartadas = 0
    for texto in textos:
        try:
            soma = soma + float(texto)
        except ValueError:
            descartadas = descartadas + 1
    return round(soma, 2), descartadas
```

</details>

## 📟 Resolvendo o chamado

O padrão completo — uma função que devolve a data da linha ou `None`, e um laço que
conta o que ficou e o que foi descartado — está no capítulo:
📖 [capítulo 8 · O padrão completo](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/#o-padrao-completo)

Rode a célula abaixo: é a coleta da noite, com três linhas problemáticas.

In [ ]:
coleta_noite = [
    "2026-03-02 22:10:05 CRITICAL OLT-CENTRO-01 perda de sinal",
    "2026-03-02 22:14:40 WARNING ONU-SUL-4512 potencia baixa",
    "2026-03-02 2#:1$:00 CRITICAL OLT-CENTRO-01 relogio corrompido",
    "",
    "2026-03-02 23:02:11 INFO OLT-CENTRO-01 alarme encerrado",
    "linha cortada no meio",
    "2026-03-02 23:40:05 CRITICAL SWITCH-NORTE-02 porta 3 down",
]

### 🎯 Sua vez — O resumo que não morre

A função `momento_da_linha` já está escrita (é a do capítulo). Escreva
`resumo_coleta(linhas)`, que devolve **três valores**: quantas linhas válidas, quantas
descartadas, e os **minutos** entre o primeiro e o último momento válido
(arredondados com 1 casa). Se nenhuma linha for válida, os minutos são `0`.

In [ ]:
from datetime import datetime


def momento_da_linha(linha):
    """Devolve o datetime da linha, ou None se ela não estiver no formato."""
    campos = linha.split()
    if len(campos) < 5:
        return None
    try:
        return datetime.strptime(f"{campos[0]} {campos[1]}", "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None


def resumo_coleta(linhas):
    # sua solução aqui
    pass

In [ ]:
confere(resumo_coleta, [
    ((coleta_noite,), (4, 3, 90.0)),
    ((["", "lixo"],), (0, 2, 0)),
    ((["2026-03-02 10:00:00 INFO A ok"],), (1, 0, 0.0)),
])

<details>
<summary><b>💡 Dica</b></summary>

Guarde os momentos válidos numa lista e conte os `None` como descartados. Os minutos
são `max(momentos) - min(momentos)` — mas só se a lista não estiver vazia (teste com
`if len(momentos) == 0`).

</details>

<details>
<summary><b>✅ Uma solução</b></summary>

```python
from datetime import datetime


def momento_da_linha(linha):
    """Devolve o datetime da linha, ou None se ela não estiver no formato."""
    campos = linha.split()
    if len(campos) < 5:
        return None
    try:
        return datetime.strptime(f"{campos[0]} {campos[1]}", "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None


def resumo_coleta(linhas):
    momentos = []
    descartadas = 0
    for linha in linhas:
        momento = momento_da_linha(linha)
        if momento is None:
            descartadas = descartadas + 1
            continue
        momentos.append(momento)
    if len(momentos) == 0:
        return 0, descartadas, 0
    duracao = max(momentos) - min(momentos)
    return len(momentos), descartadas, round(duracao.total_seconds() / 60, 1)
```

</details>

**Resposta ao chamado:** 4 linhas válidas, 3 descartadas **e informadas**, e uma
janela de alarmes de 90 minutos. A linha com o relógio corrompido não derruba mais o
resumo — e o número de descartadas no relatório avisa que algo no coletor precisa de
atenção.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** É JSON válido:
a) `{'nome': 'OLT-A'}`  b) `{"ativo": True}`  c) `{"ativo": true}`  d) `{nome: "OLT-A"}`

<details>
<summary><b>Resposta da 1</b></summary>

**c**. Aspas duplas e booleano em minúsculas.

</details>

**2.** Subtrair dois `datetime` devolve:
a) um número de segundos  b) um `timedelta`  c) uma string  d) erro

<details>
<summary><b>Resposta da 2</b></summary>

**b** — e `.total_seconds()` o transforma em número.

</details>

**3.** O `try/except` que envolve o **laço inteiro**, num arquivo cuja terceira
linha está malformada:
a) processa tudo  b) processa até a segunda linha e para  c) pula só a terceira
d) não processa nada

<details>
<summary><b>Resposta da 3</b></summary>

**b**. Para pular só a linha ruim, o `try` vai **dentro** do laço.

</details>

## 🏠 Para casa

- [Lista 08](https://lacouth.github.io/python_telecom-site/listas/lista08/) — JSON,
  datas e exceções, com testes automáticos no Colab.
- Releia o [capítulo 8 do site](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/), principalmente "O padrão completo": é o esqueleto do
  projeto final.
- **Na próxima aula:** mini-teste sobre esta aula (JSON, `strptime` e a posição do
  `try`).